In [ ]:
def longest_unique_substring(s):
    temp = {}
    start = 0
    max_len = 0
    large_str = ""
    for end in range(len(s)):
        if s[end] in temp and temp[s[end]] >= start:
            start = temp[s[end]] + 1
        temp[s[end]] = end
        cur = end - start + 1
        if cur > max_len:
            max_len = cur
            large_str = s[start : end+ 1]

    return large_str


# Example usage:
s = "asfdsafsadewrwrdddsdssf"
print(longest_unique_substring(s))

In [ ]:

1. How do decorators work in Python?
Explain the purpose of decorators, how they wrap functions,
and provide an example of a custom decorator. Also describe real-world use cases such as authentication or logging.



    A decorator is a function that takes another function as input, adds extra behavior
without modifying the original function’s code, and returns a new function.

    👉 Decorators wrap one function inside another.
    👉 They extend functionality (before/after) without changing original logic.

        Decorators are widely used in:
        
        Authentication / Authorization
        Logging & Audit Trails
        Performance Monitoring
        Rate Limiting
        Caching
        Retry Mechanisms
        Input Validation
        Database Transactions

In [ ]:
def log(func):
    def wrapper(*args, **kwargs):
        print(f"[LOG] Called {func.__name__} with args={args} kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper

@log
def transfer_money(amount, to_account):
    return f"Transferred {amount} to {to_account}"


In [ ]:
2. What is the typical workflow of an AWS Lambda function integrated with API Gateway?

1. 1️⃣ Client Sends HTTP Request
    2. API Gateway receives the request.
    3. API Gateway Validates and Routes the Request

API Gateway performs:

    Authentication/Authorization (IAM, Cognito, JWT, API Keys)
    Throttling & Rate Limiting
    Request validation (Query params, headers, body)
    Mapping templates (if using REST API)
    CORS handling

Routing to correct resource + method

Then API Gateway converts the incoming HTTP request into a JSON event object.

    4. 3️⃣ API Gateway Invokes AWS Lambda


API Gateway calls your Lambda function with the generated event:

5. Lambda Executes Your Code
6. 5️⃣ Lambda Sends a Response Back to API Gateway
7. 6️⃣ API Gateway Converts Lambda Output → HTTP Response

Client → API Gateway → Request Validation → Event → Lambda → JSON Response →
API Gateway transforms response → Client


In [ ]:

5. Have you used FastAPI? If yes, explain how you would deploy a FastAPI application behind API Gateway using AWS Lambda.
(Optional: explain how request handling works using Django/FastAPI or any Python framework.)

1. 1️⃣ Write FastAPI application
@app.get("/hello")
def hello():
    return {"message": "Hello from Lambda"}

# Lambda entry point
handler = Mangum(app)

2️⃣ Use Mangum Adapter

Mangum converts the AWS API Gateway → Lambda event into ASGI format, so FastAPI can understand the request.

3️⃣ Deploy using any IaC

4️⃣ API Gateway → Lambda Request Flow

Client
   ↓
API Gateway
   ↓
Converts HTTP request → event dictionary
   ↓
Lambda (handler)
   ↓ Mangum converts event → ASGI request
FastAPI processes route normally
   ↓ Mangum converts ASGI response → Lambda response
API Gateway
   ↓
Client receives HTTP response


In [ ]:
3. In MongoDB, if a collection has millions of records and a query using $sort and $group is slow,
how would you troubleshoot and optimize it?
Discuss indexing strategies, compound indexes, using explain plans, query patterns, aggregation optimization, etc.

    1️⃣ Check the Query Using explain()
    explain("executionStats") runs the query, chooses the best query plan, executes it, 
    and returns internal statistics like index usage, documents scanned, keys scanned, and total execution time. 
    It helps understand exactly how MongoDB executed the query and whether it used indexes or performed a collection scan.
    2️⃣ Add the RIGHT Index
    3️⃣ Compound Index Strategy
    5️⃣ Reduce the Dataset Before $group
    6️⃣ Optimize $sort Stage
7️⃣ Optimize $group Stage

In [ ]:
5. Have you used FastAPI? If yes, explain how you would deploy a FastAPI application behind API Gateway using AWS Lambda.
(Optional: explain how request handling works using Django/FastAPI or any Python framework.)


    Client → API Gateway → Lambda (FastAPI + Mangum) → DynamoDB/RDS/S3


1. Create a fastapi applications

        from fastapi import FastAPI
        from mangum import Mangum
        
        app = FastAPI()
        
        @app.get("/hello")
        def say_hello():
            return {"message": "Hello from FastAPI on Lambda!"}
        
        handler = Mangum(app)

        AWS Lambda does not understand FastAPI or ASGI apps directly.
        Mangum is an adapter that converts:
        API Gateway events → ASGI request
        and
        ASGI response → API Gateway response

2. Package Dependencies
    For AWS Lambda, you must package:
     Zip the package OR use Lambda layers.

3. Create a Lambda Function
4. Configure API Gateway

Choose API Gateway → Lambda Proxy Integration.

5. Test the API

API Gateway will hit Lambda → Mangum → FastAPI → Response.

    🌟 What Happens Internally (Request Flow Explanation)

    🔹 Step 1: Client Request
    
         Client → API Gateway
               GET /hello

   🔹 Step 2: API Gateway Converts HTTP → Lambda Event
    API Gateway does NOT forward raw HTTP.
         Instead, it sends a JSON event:
{
  "httpMethod": "GET",
  "path": "/hello",
  "headers": {},
  "body": null,
  "isBase64Encoded": false
}

🔹 Step 3: Lambda Receives Event + Context
AWS Lambda calls your function:
🔹 Step 4: Mangum Converts Lambda Event → ASGI Scope

🔹 Step 5: FastAPI Processes Request

    FastAPI:
        Validates request with Pydantic
        Executes endpoint function
        Builds response object

🔹 Step 6: Response Goes Back

FastAPI → Mangum → Lambda → API Gateway → Client.